
# 🌐 03 - Aplicação Web (Flask) com Login

Esta aplicação web permite:

- Login e logout de usuários cadastrados na tabela `usuarios`
- Cadastro de novas pessoas acolhidas (tabela `pessoas`)
- Listagem básica das pessoas cadastradas

## Ordem recomendada de uso

1. `00_mysql_config.ipynb` → salva a configuração do MySQL.
2. `01_config_db.ipynb` → cria as tabelas `pessoas` e `usuarios`.
3. `02_funcoes_cadastro.ipynb` → funções de apoio para `pessoas` (opcional).
4. Este notebook → sobe o servidor Flask.

> Antes de rodar, instale as dependências (no terminal do ambiente Python):
>
> ```bash
> pip install flask mysql-connector-python
> ```


In [4]:
from flask import (
    Flask,
    request,
    redirect,
    url_for,
    session,
    flash,
    render_template_string,
)
from werkzeug.security import generate_password_hash, check_password_hash
from functools import wraps
import mysql.connector
from pathlib import Path
import json
from datetime import datetime

# ============================================================
# 🔧 Carrega configuração do MySQL a partir do mysql_config.json
# (gerado pelo 00_mysql_config.ipynb)
# ============================================================

CONFIG_PATH = Path("mysql_config.json")


def carregar_config_mysql() -> dict:
    if not CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"Arquivo de configuração {CONFIG_PATH} não encontrado.\n"
            "Execute antes o notebook 00_mysql_config.ipynb para gerar o mysql_config.json."
        )
    return json.loads(CONFIG_PATH.read_text(encoding="utf-8"))


def get_connection():
    cfg = carregar_config_mysql()
    return mysql.connector.connect(
        host=cfg["host"],
        port=cfg["port"],
        user=cfg["user"],
        password=cfg["password"],
        database=cfg["database"],
    )


# ============================================================
# 🌐 App Flask
# ============================================================

app = Flask(__name__)
# IMPORTANTE: troque essa chave depois por algo forte e secreto
app.secret_key = "mude-esta-chave-para-uma-string-bem-grande-e-secreta"


# ============================================================
# 👥 Funções auxiliares de usuário (tabela usuarios)
# ============================================================

def criar_usuario(nome: str, email: str, senha: str, perfil: str = "colaborador"):
    conn = get_connection()
    cur = conn.cursor()

    senha_hash = generate_password_hash(senha)

    sql = """
    INSERT INTO usuarios (nome, email, senha_hash, perfil, ativo)
    VALUES (%s, %s, %s, %s, 1)
    """
    cur.execute(sql, (nome, email, senha_hash, perfil))
    conn.commit()

    cur.close()
    conn.close()


def buscar_usuario_por_email(email: str):
    conn = get_connection()
    cur = conn.cursor(dictionary=True)

    sql = "SELECT * FROM usuarios WHERE email = %s AND ativo = 1"
    cur.execute(sql, (email,))
    row = cur.fetchone()

    cur.close()
    conn.close()
    return row


# ============================================================
# 🔐 Decorator para rotas que exigem login
# ============================================================

def login_required(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        if "usuario_id" not in session:
            flash("Faça login para acessar essa página.", "warning")
            return redirect(url_for("login"))
        return f(*args, **kwargs)

    return wrapper


# ============================================================
# 🧱 Layout base (template HTML)
# ============================================================

layout_base = """
<!doctype html>
<html lang="pt-br">
<head>
    <meta charset="utf-8">
    <title>{{ titulo or "Um novo lar" }}</title>
    <style>
        :root {
            --primary: #0d6efd;
            --primary-light: #e7f1ff;
            --accent: #4dabf7;
            --bg: #f4f7fb;
            --text-main: #1f2933;
            --text-muted: #6b7785;
            --border-soft: #e0e6ed;
            --danger: #e03131;
            --success: #2b8a3e;
            --warning: #f08c00;
        }

        * {
            box-sizing: border-box;
        }

        body {
            font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
            margin: 0;
            padding: 24px;
            background: radial-gradient(circle at top left, #eef5ff 0, #f4f7fb 40%, #edf2ff 100%);
            color: var(--text-main);
        }

        .app-shell {
            max-width: 1100px;
            margin: 0 auto;
            background: #ffffff;
            border-radius: 18px;
            padding: 20px 24px 24px;
            box-shadow:
                0 22px 45px rgba(15, 23, 42, 0.08),
                0 0 0 1px rgba(15, 23, 42, 0.02);
        }

        .topbar {
            display: flex;
            align-items: center;
            justify-content: space-between;
            gap: 16px;
            margin-bottom: 20px;
        }

        .brand {
            display: flex;
            align-items: center;
            gap: 12px;
        }

        .brand-logo {
            width: 52px;
            height: 52px;
            border-radius: 16px;
            overflow: hidden;
            background: radial-gradient(circle at 20% 20%, #ffffff 0, #dbeafe 40%, #bfdbfe 100%);
            display: flex;
            align-items: center;
            justify-content: center;
            box-shadow: 0 12px 25px rgba(37, 99, 235, 0.35);
        }

        .brand-logo img {
            max-width: 70%;
            max-height: 70%;
            display: block;
        }

        .brand-text-main {
            font-weight: 700;
            letter-spacing: 0.04em;
            text-transform: uppercase;
            font-size: 13px;
            color: var(--primary);
        }

        .brand-text-sub {
            font-size: 12px;
            color: var(--text-muted);
        }

        .page-title {
            font-weight: 600;
            font-size: 18px;
            color: var(--text-main);
            margin: 2px 0 0;
        }

        .menu {
            display: flex;
            flex-wrap: wrap;
            align-items: center;
            gap: 8px;
        }

        .menu a {
            font-size: 13px;
            padding: 6px 10px;
            border-radius: 999px;
            text-decoration: none;
            color: var(--text-muted);
            background: #f8fafc;
            border: 1px solid rgba(148, 163, 184, 0.35);
            transition: all 0.15s ease-out;
        }

        .menu a:hover {
            color: var(--primary);
            border-color: rgba(37, 99, 235, 0.6);
            background: #eff6ff;
        }

        .content-card {
            margin-top: 4px;
            padding: 16px 18px 18px;
            border-radius: 14px;
            background: linear-gradient(135deg, #ffffff 0%, #f9fbff 50%, #ffffff 100%);
            border: 1px solid rgba(226, 232, 240, 0.9);
        }

        table {
            width: 100%;
            border-collapse: collapse;
            margin-top: 8px;
            border-radius: 12px;
            overflow: hidden;
            font-size: 13px;
        }

        thead {
            background: linear-gradient(90deg, #e7f1ff, #dee9ff);
        }

        th, td {
            padding: 8px 10px;
            border-bottom: 1px solid #edf2ff;
        }

        th {
            text-align: left;
            font-weight: 600;
            color: #475569;
        }

        tr:nth-child(even) td {
            background: #f9fbff;
        }

        tr:hover td {
            background: #eef3ff;
        }

        .flash {
            padding: 8px 10px;
            margin-bottom: 10px;
            border-radius: 8px;
            font-size: 13px;
            border: 1px solid transparent;
            display: flex;
            align-items: center;
            gap: 8px;
        }

        .flash-success {
            background: #edf9f0;
            color: var(--success);
            border-color: rgba(34, 197, 94, 0.4);
        }

        .flash-warning {
            background: #fff7e6;
            color: var(--warning);
            border-color: rgba(245, 158, 11, 0.4);
        }

        .flash-error {
            background: #ffe8e8;
            color: var(--danger);
            border-color: rgba(248, 113, 113, 0.5);
        }

        .btn {
            display: inline-flex;
            align-items: center;
            justify-content: center;
            gap: 6px;
            padding: 7px 12px;
            border-radius: 999px;
            border: none;
            cursor: pointer;
            text-decoration: none;
            font-size: 13px;
            font-weight: 500;
            transition: all 0.16s ease-out;
        }

        .btn-primary {
            background: linear-gradient(135deg, #2563eb, #1d4ed8);
            color: #ffffff;
            box-shadow: 0 10px 25px rgba(37, 99, 235, 0.35);
            border: 1px solid rgba(30, 64, 175, 0.9);
        }

        .btn-primary:hover {
            transform: translateY(-1px);
            box-shadow: 0 14px 35px rgba(37, 99, 235, 0.4);
            background: linear-gradient(135deg, #1d4ed8, #1d4ed8);
        }

        .btn-secondary {
            background: #f8fafc;
            color: #0f172a;
            border: 1px solid rgba(148, 163, 184, 0.7);
        }

        .btn-secondary:hover {
            background: #e2e8f0;
        }

        .field-group {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
            gap: 12px 16px;
            margin-bottom: 10px;
        }

        .field {
            margin-bottom: 4px;
        }

        label {
            display: block;
            font-weight: 600;
            margin-bottom: 4px;
            font-size: 12px;
            color: #475569;
        }

        input[type=text],
        input[type=password],
        input[type=date],
        textarea,
        select {
            width: 100%;
            padding: 7px 9px;
            border-radius: 10px;
            border: 1px solid #d0d7e2;
            font-size: 13px;
            transition: all 0.16s ease-out;
            background: #ffffff;
        }

        textarea {
            min-height: 70px;
            resize: vertical;
        }

        input:focus,
        textarea:focus,
        select:focus {
            outline: none;
            border-color: rgba(37, 99, 235, 0.85);
            box-shadow: 0 0 0 1px rgba(37, 99, 235, 0.25), 0 0 0 4px rgba(191, 219, 254, 0.6);
        }

        .form-actions {
            margin-top: 10px;
            display: flex;
            flex-wrap: wrap;
            gap: 8px;
        }

        .page-header {
            margin-bottom: 6px;
        }

        .page-header h2 {
            margin: 0;
            font-size: 18px;
            font-weight: 600;
            color: #111827;
        }

        .page-header p {
            margin: 2px 0 0;
            font-size: 12px;
            color: var(--text-muted);
        }

        @media (max-width: 640px) {
            body {
                padding: 14px;
            }

            .app-shell {
                padding: 16px 14px 18px;
            }

            .topbar {
                flex-direction: column;
                align-items: flex-start;
            }

            .menu {
                width: 100%;
                justify-content: flex-start;
            }
        }
    </style>
</head>
<body>
<div class="app-shell">
    <div class="topbar">
        <div class="brand">
            <div class="brand-logo">
                <!-- Substitua 'logo_um_novo_lar.png' pelo nome real do arquivo dentro da pasta /static -->
                <img src="{{ url_for('static', filename='logo_um_novo_lar.png') }}" alt="Logo Um novo lar">
            </div>
            <div>
                <div class="brand-text-main">UM NOVO LAR</div>
                <div class="page-title">Sistema de Acolhimento</div>
                <div class="brand-text-sub">Cadastro e acompanhamento de pessoas em situação de vulnerabilidade</div>
            </div>
        </div>
        <div class="menu">
            {% if session.get('usuario_id') %}
                <a href="{{ url_for('lista_pessoas') }}">Pessoas acolhidas</a>
                <a href="{{ url_for('nova_pessoa') }}">Novo cadastro</a>
                <a href="{{ url_for('logout') }}">Sair</a>
            {% else %}
                <a href="{{ url_for('login') }}">Login</a>
                <a href="{{ url_for('registrar') }}">Criar acesso</a>
            {% endif %}
        </div>
    </div>

    {% with messages = get_flashed_messages(with_categories=True) %}
      {% if messages %}
        {% for cat, msg in messages %}
          <div class="flash flash-{{ cat }}">{{ msg }}</div>
        {% endfor %}
      {% endif %}
    {% endwith %}

    <div class="content-card">
        {{ conteudo|safe }}
    </div>
</div>
</body>
</html>
"""


def render_page(titulo: str, conteudo_html: str):
    return render_template_string(layout_base, titulo=titulo, conteudo=conteudo_html)


# ============================================================
# 🧭 Rotas
# ============================================================

@app.route("/")
def index():
    if "usuario_id" in session:
        return redirect(url_for("lista_pessoas"))
    return redirect(url_for("login"))


# ---------- Login ----------
@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        email = request.form.get("email", "").strip().lower()
        senha = request.form.get("senha", "")

        usuario = buscar_usuario_por_email(email)
        if not usuario or not check_password_hash(usuario["senha_hash"], senha):
            flash("E-mail ou senha inválidos.", "error")
        else:
            session["usuario_id"] = usuario["id"]
            session["usuario_nome"] = usuario["nome"]
            flash("Login realizado com sucesso.", "success")
            return redirect(url_for("lista_pessoas"))

    conteudo = """
    <h2>Login</h2>
    <form method="post">
        <div class="field">
            <label>E-mail</label>
            <input type="text" name="email" required>
        </div>
        <div class="field">
            <label>Senha</label>
            <input type="password" name="senha" required>
        </div>
        <button type="submit" class="btn btn-primary">Entrar</button>
    </form>
    """
    return render_page("Login", conteudo)


# ---------- Logout ----------
@app.route("/logout")
def logout():
    session.clear()
    flash("Você saiu do sistema.", "success")
    return redirect(url_for("login"))


# ---------- Cadastro de usuário (registro) ----------
@app.route("/registrar", methods=["GET", "POST"])
def registrar():
    if request.method == "POST":
        nome = request.form.get("nome", "").strip()
        email = request.form.get("email", "").strip().lower()
        senha = request.form.get("senha", "")
        senha2 = request.form.get("senha2", "")

        if not nome or not email or not senha:
            flash("Preencha nome, e-mail e senha.", "warning")
        elif senha != senha2:
            flash("As senhas não conferem.", "warning")
        else:
            existente = buscar_usuario_por_email(email)
            if existente:
                flash("Já existe um usuário ativo com esse e-mail.", "warning")
            else:
                criar_usuario(nome, email, senha, perfil="colaborador")
                flash("Usuário criado com sucesso. Agora você pode fazer login.", "success")
                return redirect(url_for("login"))

    conteudo = """
    <h2>Criar usuário</h2>
    <form method="post">
        <div class="field">
            <label>Nome</label>
            <input type="text" name="nome" required>
        </div>
        <div class="field">
            <label>E-mail</label>
            <input type="text" name="email" required>
        </div>
        <div class="field">
            <label>Senha</label>
            <input type="password" name="senha" required>
        </div>
        <div class="field">
            <label>Confirme a senha</label>
            <input type="password" name="senha2" required>
        </div>
        <button type="submit" class="btn btn-primary">Salvar</button>
    </form>
    """
    return render_page("Cadastro de usuário", conteudo)


# ---------- Lista de pessoas ----------
@app.route("/pessoas")
@login_required
def lista_pessoas():
    conn = get_connection()
    cur = conn.cursor(dictionary=True)
    cur.execute("SELECT * FROM pessoas ORDER BY id DESC")
    pessoas = cur.fetchall()
    cur.close()
    conn.close()

    linhas = []
    for p in pessoas:
        linhas.append(f"""
            <tr>
                <td>{p['id']}</td>
                <td>{p.get('nome') or ''}</td>
                <td>{p.get('apelido') or ''}</td>
                <td>{p.get('telefone') or ''}</td>
                <td>{p.get('status') or ''}</td>
            </tr>
        """)

    link_nova_pessoa = url_for("nova_pessoa")

    conteudo = f"""
    <h2>Pessoas acolhidas</h2>
    <p><a class="btn btn-primary" href="{link_nova_pessoa}">Novo cadastro</a></p>
    <table>
        <thead>
            <tr>
                <th>ID</th>
                <th>Nome</th>
                <th>Apelido</th>
                <th>Telefone</th>
                <th>Status</th>
            </tr>
        </thead>
        <tbody>
            {'\\n'.join(linhas)}
        </tbody>
    </table>
    """
    return render_page("Pessoas", conteudo)


# ---------- Nova pessoa ----------
@app.route("/pessoas/nova", methods=["GET", "POST"])
@login_required
def nova_pessoa():
    if request.method == "POST":
        nome = request.form.get("nome", "").strip()
        apelido = request.form.get("apelido", "").strip() or None
        data_nascimento = request.form.get("data_nascimento") or None
        documento_principal = request.form.get("documento_principal", "").strip() or None
        tem_documentos = request.form.get("tem_documentos") == "on"
        telefone = request.form.get("telefone", "").strip() or None
        contato_emergencia = request.form.get("contato_emergencia", "").strip() or None
        cidade_origem = request.form.get("cidade_origem", "").strip() or None
        situacao_rua_desde = request.form.get("situacao_rua_desde", "").strip() or None
        saude_resumo = request.form.get("saude_resumo", "").strip() or None
        dependencias_quimicas = request.form.get("dependencias_quimicas", "").strip() or None
        observacoes = request.form.get("observacoes", "").strip() or None

        if not nome:
            flash("Nome é obrigatório.", "warning")
        else:
            conn = get_connection()
            cur = conn.cursor()

            insert_sql = """
            INSERT INTO pessoas (
                nome,
                apelido,
                data_nascimento,
                documento_principal,
                tem_documentos,
                telefone,
                contato_emergencia,
                cidade_origem,
                situacao_rua_desde,
                saude_resumo,
                dependencias_quimicas,
                observacoes,
                status,
                data_cadastro
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            tem_docs_int = 1 if tem_documentos else 0
            valores = (
                nome,
                apelido,
                data_nascimento or None,
                documento_principal,
                tem_docs_int,
                telefone,
                contato_emergencia,
                cidade_origem,
                situacao_rua_desde,
                saude_resumo,
                dependencias_quimicas,
                observacoes,
                "ativo",
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            )

            cur.execute(insert_sql, valores)
            conn.commit()
            cur.close()
            conn.close()

            flash("Pessoa cadastrada com sucesso.", "success")
            return redirect(url_for("lista_pessoas"))

    link_lista = url_for("lista_pessoas")

    conteudo = f"""
    <h2>Novo cadastro de pessoa acolhida</h2>
    <form method="post">
        <div class="field">
            <label>Nome completo *</label>
            <input type="text" name="nome" required>
        </div>
        <div class="field">
            <label>Apelido</label>
            <input type="text" name="apelido">
        </div>
        <div class="field">
            <label>Data de nascimento</label>
            <input type="date" name="data_nascimento">
        </div>
        <div class="field">
            <label>Documento principal (RG/CPF ou outro)</label>
            <input type="text" name="documento_principal">
        </div>
        <div class="field">
            <label><input type="checkbox" name="tem_documentos"> Possui documentos básicos</label>
        </div>
        <div class="field">
            <label>Telefone</label>
            <input type="text" name="telefone">
        </div>
        <div class="field">
            <label>Contato de emergência</label>
            <input type="text" name="contato_emergencia">
        </div>
        <div class="field">
            <label>Cidade de origem</label>
            <input type="text" name="cidade_origem">
        </div>
        <div class="field">
            <label>Em situação de rua desde quando?</label>
            <textarea name="situacao_rua_desde" rows="2"></textarea>
        </div>
        <div class="field">
            <label>Resumo da situação de saúde</label>
            <textarea name="saude_resumo" rows="3"></textarea>
        </div>
        <div class="field">
            <label>Dependências químicas</label>
            <textarea name="dependencias_quimicas" rows="2"></textarea>
        </div>
        <div class="field">
            <label>Observações gerais</label>
            <textarea name="observacoes" rows="3"></textarea>
        </div>
        <button type="submit" class="btn btn-primary">Salvar</button>
        <a href="{link_lista}" class="btn btn-secondary">Voltar</a>
    </form>
    """
    return render_page("Nova pessoa", conteudo)


# ============================================================
# ▶️ Rodar app
# ============================================================

if __name__ == "__main__":
    # Em notebook, é melhor sem reloader
    app.run(
        debug=True,
        host="0.0.0.0",
        port=5000,
        use_reloader=False,
    )


SyntaxError: f-string expression part cannot include a backslash (4037035895.py, line 312)